In [ ]:
!pip install datasets
import  datasets

1st code cell is the main one. The 2nd and third were just for continuing from previous checkpoints.

In [ ]:
import pandas as pd
import pickle
from transformers import BertTokenizer
from transformers import BertModel
import torch
import optuna
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset


print("Cleaning of the corpus")
train_df= pd.read_json('corpus.json')

print(train_df.shape)
print(type(train_df))
# Checking the fields for text and labels
print(train_df.columns)

text_field = "text"
label_field = "classificationCategory"
if text_field in train_df.columns and label_field in train_df.columns:
    print("Text and Label fields are correctly identified.")
else:
    print("Error: Fields not found!")

# Count missing values per column
print(train_df.isnull().sum())

#to find and drop duplicate entries

print(f"Original dataset size: {len(train_df)}")
train_df = train_df.map(lambda x: str(x) if isinstance(x, (dict, list)) else x)
train_df = train_df.drop_duplicates()
print(f"Dataset size after removing duplicates: {len(train_df)}")


# Tokenize the text for training the model

train_df['combined'] = train_df['title'] + " " + train_df['text']
texts = train_df['combined'].tolist()
category = train_df['classificationCategory']


#encode labels
label_encoder = LabelEncoder()
train_df['labels'] = label_encoder.fit_transform(category)

# Number of labels corresponds to the unique categories
num_labels = len(label_encoder.classes_)

# Split dataset into train+validation and test sets
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    train_df["combined"].tolist(),
    train_df["labels"].tolist(),
    test_size=0.2,  # Reserve 20% of the total data for testing
    random_state=42
)

# Further split train+validation into train and validation sets
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts,
    train_val_labels,
    test_size=0.25,  # Reserve 25% of train+validation for validation (0.25 * 0.8 = 0.2 total data)
    random_state=42
)

# Load pretrained tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize datasets
train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=128)
val_encodings = tokenizer(val_texts, padding=True, truncation=True, max_length=128)
test_encodings = tokenizer(test_texts, padding=True, truncation=True, max_length=128)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

val_dataset = Dataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'labels': val_labels
})

test_dataset = Dataset.from_dict({

})

# Load the model

def model_init():
    """
    This function is called during hyperparameter search to initialize a new model instance.
    It returns an instance of AutoModelForSequenceClassification with the specified number of labels.
    """
    return AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels= num_labels)

# Define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

# Define training arguments
training_args = TrainingArguments(
   output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none"
)

# Define the Trainer
trainer = Trainer(
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,# Include validation dataset
    model_init=model_init,
    compute_metrics=compute_metrics  # Explicitly pass compute_metrics to the Trainer
)

# Define the hyperparameter search space
def hp_space_optuna(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 5),
        "weight_decay": trial.suggest_float("weight_decay", 0.01, 0.3),
    }

# Perform hyperparameter search with Optuna
best_run = trainer.hyperparameter_search(
    direction="maximize",  # Maximize metric (e.g., accuracy, F1 score)
    backend="optuna",
    n_trials=10,  # Number of trials to find the best hyperparameters
    hp_space=hp_space_optuna
)

# Print the best hyperparameters
print("Best Hyperparameters:", best_run.hyperparameters)

# Update training arguments with the best hyperparameters
for n, v in best_run.hyperparameters.items():
    setattr(trainer.args, n, v)

# Train with the best Hyperparameters
trainer.train()


# Evaluate the model
metrics = trainer.evaluate()
print("Evaluation Metrics:")
print(metrics)

# Save the final model
trainer.save_model("./best_model")

# Evaluate the model
eval_results = trainer.evaluate()
print("Validation Results:", eval_results)

# Test the model
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
true_labels = test_labels

# Classification report
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, target_names=label_encoder.classes_))

# Confusion matrix
conf_matrix = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()










In [ ]:
# Install required packages
!pip install transformers datasets sklearn torch seaborn

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import logging
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Set device and check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU Model: {torch.cuda.get_device_name(0)}")
    logger.info(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

def load_and_prepare_data(data_path):
    """Load and prepare the dataset"""
    logger.info("Loading and preparing data...")
    train_df = pd.read_json(data_path)

    # Combine title and text with proper handling of NaN values
    train_df['combined'] = train_df['title'].fillna('') + " " + train_df['text'].fillna('')

    # Clean text
    train_df['combined'] = train_df['combined'].str.strip()
    train_df = train_df[train_df['combined'].str.len() > 0]

    return train_df

def create_datasets(train_df, tokenizer):
    """Create train, validation, and test datasets"""
    # Encode labels
    label_encoder = LabelEncoder()
    train_df['labels'] = label_encoder.fit_transform(train_df['classificationCategory'])
    num_labels = len(label_encoder.classes_)

    # Split dataset with stratification
    train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
        train_df["combined"].tolist(),
        train_df["labels"].tolist(),
        test_size=0.2,
        random_state=42,
        stratify=train_df["labels"]
    )

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_val_texts,
        train_val_labels,
        test_size=0.25,
        random_state=42,
        stratify=train_val_labels
    )

    # Tokenize with dynamic padding
    max_length = min(512, max(len(t.split()) for t in train_texts))
    logger.info(f"Using max length of {max_length} tokens")

    def tokenize_function(texts):
        return tokenizer(
            texts,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

    # Create datasets
    datasets = {
        'train': Dataset.from_dict({
            **tokenize_function(train_texts),
            'labels': train_labels
        }),
        'validation': Dataset.from_dict({
            **tokenize_function(val_texts),
            'labels': val_labels
        }),
        'test': Dataset.from_dict({
            **tokenize_function(test_texts),
            'labels': test_labels
        })
    }

    return datasets, label_encoder, num_labels

def compute_metrics(eval_pred):
    """Compute evaluation metrics"""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

def main():
    # Define paths for Google Drive
    DRIVE_PATH = "/content/drive/MyDrive"  # Update this to your Drive folder
    os.makedirs(DRIVE_PATH, exist_ok=True)

    # Set paths
    checkpoint_path = os.path.join(DRIVE_PATH, "checkpoint-540")
    output_dir = os.path.join(DRIVE_PATH, "continued_training")
    data_path = os.path.join(DRIVE_PATH, "corpus.json")

    os.makedirs(output_dir, exist_ok=True)

    # Load data
    train_df = load_and_prepare_data(data_path)

    # Load tokenizer and model from checkpoint
    logger.info("Loading model and tokenizer from checkpoint...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)

    # Create datasets
    datasets, label_encoder, num_labels = create_datasets(train_df, tokenizer)

    # Define Colab-optimized training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        evaluation_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        learning_rate=5e-6,
        per_device_train_batch_size=8,  # Reduced for Colab GPU memory
        per_device_eval_batch_size=16,
        num_train_epochs=2,
        weight_decay=0.01,
        logging_dir=os.path.join(DRIVE_PATH, "logs"),
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        save_total_limit=2,
        resume_from_checkpoint=True,
        overwrite_output_dir=True,
        warmup_steps=100,
        fp16=True if torch.cuda.is_available() else False,
        gradient_accumulation_steps=4,  # Increased for effective batch size
        report_to="none",
        # Colab-specific optimizations
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
    )

    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=datasets['train'],
        eval_dataset=datasets['validation'],
        compute_metrics=compute_metrics,
    )

    # Resume training
    logger.info("Resuming training from checkpoint...")
    trainer.train(resume_from_checkpoint=checkpoint_path)

    # Evaluate
    logger.info("Evaluating model...")
    metrics = trainer.evaluate()
    logger.info(f"Evaluation Metrics: {metrics}")

    # Save metrics to Drive
    with open(os.path.join(DRIVE_PATH, 'evaluation_metrics.txt'), 'w') as f:
        f.write(str(metrics))

    # Test predictions
    logger.info("Generating test predictions...")
    predictions = trainer.predict(datasets['test'])
    pred_labels = np.argmax(predictions.predictions, axis=-1)
    true_labels = datasets['test']['labels']

    # Save results
    results_dir = os.path.join(DRIVE_PATH, "final_results")
    os.makedirs(results_dir, exist_ok=True)

    # Classification report
    class_report = classification_report(true_labels, pred_labels,
                                      target_names=label_encoder.classes_,
                                      output_dict=True)
    pd.DataFrame(class_report).to_csv(os.path.join(results_dir, "classification_report.csv"))

    # Confusion matrix
    plt.figure(figsize=(12, 10))
    conf_matrix = confusion_matrix(true_labels, pred_labels)
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.xlabel("Predicted Labels")
    plt.ylabel("True Labels")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "confusion_matrix.png"))
    plt.close()

    # Save final model to Drive
    final_model_path = os.path.join(DRIVE_PATH, "final_model")
    trainer.save_model(final_model_path)
    tokenizer.save_pretrained(final_model_path)

    logger.info("Training completed successfully!")

if __name__ == "__main__":
    main()

In [ ]:
import torch
import json
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch.nn.functional as F

# Set model checkpoint directory
checkpoint_dir = "/content/drive/MyDrive/final_model"  # Update if needed

# Load trained model and tokenizer
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Load dataset
with open("/content/drive/MyDrive/corpus.json", "r") as f:
    data = json.load(f)

# Extract texts
texts = [item["text"] for item in data]
labels = [item["classificationCategory"] for item in data]

# Batch processing settings
batch_size = 32  # Adjust batch size based on available RAM
num_batches = (len(texts) + batch_size - 1) // batch_size  # Total number of batches

predicted_labels = []
confidence_scores = []

# Process data in batches
for i in range(num_batches):
    print(f"Processing batch {i+1}/{num_batches}...")

    batch_texts = texts[i * batch_size : (i + 1) * batch_size]

    # Tokenize batch
    inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

    # Get predictions
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Raw scores before softmax

    # Compute softmax to get confidence scores
    probs = F.softmax(logits, dim=-1)

    # Store results
    predicted_labels.extend(torch.argmax(probs, dim=-1).cpu().numpy())
    confidence_scores.extend(torch.max(probs, dim=-1).values.cpu().numpy())

# Print results for first 10 samples
for i in range(10):
    print(f"Text: {texts[i][:100]}...")  # Print first 100 chars for readability
    print(f"True Label: {labels[i]}")
    print(f"Predicted Label: {predicted_labels[i]}")
    print(f"Confidence: {confidence_scores[i]:.4f}")
    print("-" * 50)

print("✅ Inference completed without crashes!")


In [ ]:
!pip install optuna

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
